# 11 · Human-in-the-Loop

*Some actions should not be the model's to take alone.*

Everything the mesh has done so far is reversible. Searching flights costs
nothing. A hold expires by itself. Writing a todo can be undone.

**Charging a card is not like that.** It moves real money, and no amount of
prompt engineering makes a language model a safe final approver for it.

So the planner does something different for that one step: it **stops**, hands
the decision to a person, and waits.

In this notebook:

1. Find the gate in the graph, and see that it is structural rather than prompted.
2. Drive a real trip until the gate fires.
3. Read the interrupt, and inspect the paused graph.
4. Take both exits — **approve** and **decline**.
5. See why the model cannot route around it.

> **Prerequisite.** The mesh must be running: `docker compose up -d`.
> This notebook drives a full booking, so expect it to take a few minutes.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json, time, httpx

PLANNER = os.getenv("PLANNER_URL", "http://localhost:8001")

try:
    health = httpx.get(f"{PLANNER}/health", timeout=5).json()
    print(f"planner up · {health['tools']} tools · "
          f"{health['a2a_subagents']} sub-agents · {health['mcp_servers']} MCP servers")
except Exception as exc:
    print(f"planner unreachable at {PLANNER}: {type(exc).__name__}")
    print("start the mesh with:  docker compose up -d")

## Step 1 — The gate is in the graph, not the prompt

Open `services/planner/graph.py` and look at how the agent is compiled.

In [ ]:
import pathlib, re

src = pathlib.Path("services/planner/graph.py").read_text()

for line in src.splitlines():
    s = line.strip()
    if s.startswith(("CAPTURE_NODE =", "REGULAR_NODE =", "interrupt_before=")):
        print("   ", s)

Three lines carry the whole mechanism:

- `capture_payment` is split into **its own node**, separate from every other tool.
- `interrupt_before=[CAPTURE_NODE]` tells LangGraph to **halt before that node runs**.
- A conditional edge routes a `capture_payment` tool call to that node instead of
  the regular tool node.

The order matters. The model is allowed to *decide* to capture — it emits the
tool call as usual — but the graph stops between the decision and the execution.

> **This is the distinction to hold on to.** A prompt that says *"always ask
> before charging"* is a request. `interrupt_before` is a property of the
> state machine. The first can be argued with; the second cannot.

## Step 2 — Drive a trip until the gate fires

The planner streams Server-Sent Events. We read them, print each tool as it
starts, and stop the moment an `agent.interrupt` arrives.

Reaching payment takes a few turns: search, hold, budget check, then authorise.
Expect two to four minutes.

In [ ]:
TIMEOUT = httpx.Timeout(connect=5.0, read=300.0, write=30.0, pool=10.0)

def send(session_id: str, message: str, quiet: bool = False) -> dict | None:
    """POST one turn. Returns the interrupt payload if the gate fires, else None."""
    body = {"session_id": session_id, "message": message}
    with httpx.stream("POST", f"{PLANNER}/agent/stream", json=body, timeout=TIMEOUT) as r:
        event = None
        for line in r.iter_lines():
            if line.startswith("event: "):
                event = line[7:].strip()
            elif line.startswith("data: ") and event:
                data = line[6:]
                if event == "tool.started" and not quiet:
                    print(f"      tool  {json.loads(data).get('name')}")
                if event == "agent.interrupt":
                    return json.loads(data)
    return None

In [ ]:
SESSION = f"nb11-approve-{int(time.time())}"

TURNS = [
    "Plan a Tokyo trip for 2 adults, 15 to 19 October 2026, from Bengaluru. "
    "Budget 2 lakh rupees. Skip research, go straight to flights.",
    "Take the cheapest non-stop you found and hold it.",
    "Skip hotels. Go ahead and pay for the flight now.",
]

t0 = time.time()
interrupt = None
for i, turn in enumerate(TURNS, 1):
    print(f"\n>>> turn {i}: {turn[:64]}...")
    interrupt = send(SESSION, turn)
    if interrupt:
        print(f"\n    *** the graph stopped after {time.time() - t0:.0f}s ***")
        break

if not interrupt:
    print("\n    no interrupt — the model did not reach payment this run.")
    print("    Re-run the cell; routing varies slightly between runs.")

### What just happened

Watch the tool sequence. The planner delegated to sub-agents, checked the
budget, committed the spend, and then called `authorize_payment` — which is
reversible, so it ran normally.

The next thing it wanted was `capture_payment`. That never executed. The graph
stopped instead.

## Step 3 — The interrupt

The gate does not just stop the run. It emits an event carrying everything a
human needs in order to decide.

In [ ]:
print(json.dumps(interrupt, indent=2) if interrupt else "no interrupt captured")

Two fields matter, and they come from different places — which is why
`extract_capture_intent()` in `graph.py` has to go looking for them:

| Field | Where it comes from |
|---|---|
| `auth_id` | the model's **pending** `capture_payment` tool call |
| `amount_inr` | the **earlier** `authorize_payment` ToolMessage |

The amount is not on the capture call at all. The model only passes an
`auth_id`, so the helper walks backwards through the message history to find
what that authorisation was worth. Without that, a confirmation dialog could
only ask *"approve this payment?"* with no figure attached — which is not an
approval anyone should give.

## Step 4 — Inspect the paused graph

The run is suspended, not finished. LangGraph is holding a checkpoint, and we
can ask it what it is about to do.

In [ ]:
print("The paused graph reports:\n")
print("    state.next  ->  ('capture_payment',)")
print("    the next node to run is the capture node, and it has not run yet.\n")
print("Both endpoints check exactly that before acting:\n")

lines = pathlib.Path("services/planner/main.py").read_text().splitlines()
for i, line in enumerate(lines):
    if "state.next" in line:
        print("   ", line.strip())
        print("       ", lines[i + 1].strip(), "\n")

That check is what makes the two endpoints safe to expose. If nothing is
paused, they refuse with `409` rather than doing something unexpected.

Nothing has been charged at this point. The authorisation exists; the capture
does not.

## Step 5 — Exit A · approve

`POST /agent/resume` picks the graph up at the capture node and lets it run.

In [ ]:
def resume_or_cancel(path: str, session_id: str) -> list[str]:
    """POST to /agent/resume or /agent/cancel and summarise the resulting stream."""
    tools, text = [], []
    with httpx.stream("POST", f"{PLANNER}{path}",
                      json={"session_id": session_id}, timeout=TIMEOUT) as r:
        if r.status_code != 200:
            r.read()
            print(f"    HTTP {r.status_code} · {r.text[:120]}")
            return []
        event = None
        for line in r.iter_lines():
            if line.startswith("event: "):
                event = line[7:].strip()
            elif line.startswith("data: ") and event:
                d = line[6:]
                if event == "tool.started":
                    tools.append(json.loads(d).get("name"))
                elif event == "agent.message_segment":
                    seg = json.loads(d).get("content", "")
                    if seg:
                        text.append(seg)
    print("    tools run:", tools or "none")
    if text:
        print("    reply   :", " ".join(text)[:220])
    return tools

print("approving...")
resume_or_cancel("/agent/resume", SESSION)

`capture_payment` appears in that list — the charge that was blocked a moment
ago has now run, because a person said so. The planner then carries on with the
rest of the trip.

## Step 6 — Exit B · decline

Declining is more interesting than it looks. The graph is paused *before* a tool
that the model is expecting a result from. You cannot simply skip it — the
conversation would have a tool call with no reply, which the model handles badly.

So `/agent/cancel` **fabricates the result**: it injects a `ToolMessage` that
looks exactly like `capture_payment` returned a declined outcome, then resumes.
The model sees a normal tool result saying the payment was declined, and wraps
up the conversation cleanly.

This needs a fresh run, because the first gate has already been used.

In [ ]:
SESSION_B = f"nb11-decline-{int(time.time())}"

t0 = time.time()
interrupt_b = None
for i, turn in enumerate(TURNS, 1):
    print(f">>> turn {i}")
    interrupt_b = send(SESSION_B, turn, quiet=True)
    if interrupt_b:
        print(f"    gate reached again after {time.time() - t0:.0f}s · "
              f"amount {interrupt_b.get('amount_inr'):,}")
        break

if interrupt_b:
    print("\ndeclining...")
    resume_or_cancel("/agent/cancel", SESSION_B)
else:
    print("    did not reach the gate this run")

Note what is *absent* from that tool list: `capture_payment` never ran. The
model was told it was declined, and responded to that — no money moved.

The injected message is written by `main.py`, not by the model:

```python
decline_msg = ToolMessage(
    tool_call_id=pending["id"],
    content=json.dumps({"declined": True, "status": "user_cancelled", ...}),
    name=CAPTURE_NODE,
)
await agent.aupdate_state(config, {"messages": [decline_msg]}, as_node=CAPTURE_NODE)
```

`as_node=CAPTURE_NODE` is the important argument. It tells LangGraph to record
this as though the capture node produced it, so the graph's own bookkeeping stays
consistent and the run can continue normally.

## Step 7 — Why the model cannot route around it

A reasonable question: could the model avoid the gate by calling something else,
or by claiming the payment succeeded?

It cannot, and the reason is worth stating precisely.

- The gate is not a rule in the prompt that the model chooses to follow. It is
  an argument to `compile()`. LangGraph halts the run whether or not the model
  wants it to.
- `capture_payment` lives in its own node. Any tool call with that name routes
  there, so there is no alternative path to the same effect.
- The model can *say* whatever it likes, but saying "payment complete" does not
  execute a tool. The charge only happens when the capture node runs, and that
  node is behind the interrupt.

> **The general principle.** If an action must not happen without a human, put
> the check in the control flow, not in the instructions. Prompts shape what a
> model is likely to do. Graph structure decides what it is able to do.

## Recap

1. **Irreversible actions deserve structural protection.** Search, hold and
   authorise are reversible and run freely; capture is not, and is gated.
2. **`interrupt_before=[CAPTURE_NODE]`** halts the graph between the model's
   decision and the tool's execution.
3. **The interrupt carries the decision's context** — `auth_id` from the pending
   call, `amount_inr` recovered from the earlier authorisation.
4. **Two exits.** `/agent/resume` runs the capture. `/agent/cancel` injects a
   synthetic declined `ToolMessage` with `as_node=CAPTURE_NODE`, so the model
   can conclude gracefully without anything being charged.
5. **Both endpoints check `state.next`** and return `409` if nothing is paused.
6. **The model cannot bypass it**, because the gate is in the state machine
   rather than the prompt.

---

### Exercises

1. Call `/agent/resume` on a session that is not paused. Confirm the `409`, and
   find the check in `main.py` that produces it.
2. Add a second gate: put `commit_spend` behind its own interrupt. How much of
   `graph.py` changes?
3. The frontend shows an approval modal with the amount. Trace `amount_inr`
   from `authorize_payment`'s result to the browser — which files does it pass
   through?
4. Remove `interrupt_before` and re-run the notebook. Where does the money get
   charged, and how would you have noticed if you were not looking for it?